In [14]:
!pip install --upgrade pip setuptools wheel

In [15]:
!pip install scikit-learn==1.4.2 --only-binary=:all:

In [21]:
pip install xgboost==3.0.5

Note: you may need to restart the kernel to use updated packages.


In [16]:
import boto3
import sagemaker

In [17]:
role = sagemaker.get_execution_role()

print(role)

arn:aws:iam::472608500930:role/LabRole


In [18]:
s3 = boto3.client("s3")

response = s3.list_buckets()

for bucket in response['Buckets']:
    print(bucket['Name'])

sagemaker-us-east-1-472608500930


In [19]:
import tarfile

with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add(
        "model/best_model.joblib",
        arcname="best_model.joblib"
    )

In [20]:
s3 = boto3.client("s3")

bucket_name = "sagemaker-us-east-1-472608500930"

s3.upload_file(
    "model.tar.gz",
    bucket_name,
    "model/model.tar.gz"
)

In [22]:
from inference import model_fn

model = model_fn("model")

In [25]:
import json
import boto3
import sagemaker
from sagemaker.sklearn.model import SKLearnModel


BUCKET = "sagemaker-us-east-1-472608500930"
MODEL_S3_KEY = "model/model.tar.gz"
ENDPOINT_NAME = "credit-score-endpoint"
# -------------------------------------------------------------------------

REGION = "us-east-1"
INSTANCE_TYPE = "ml.m5.large"
FRAMEWORK_VERSION = "1.4-2"


def get_lab_role_arn() -> str:
    iam = boto3.client("iam")
    return iam.get_role(RoleName="LabRole")["Role"]["Arn"]


def main() -> None:
    boto3.setup_default_session(region_name=REGION)
    sm_session = sagemaker.Session()
    role_arn = get_lab_role_arn()
    model_s3_uri = f"s3://{BUCKET}/{MODEL_S3_KEY}"

    print(f"Role:      {role_arn}")
    print(f"Model URI: {model_s3_uri}")
    print(f"Endpoint:  {ENDPOINT_NAME}")

    model = SKLearnModel(
        model_data=model_s3_uri,
        role=role_arn,
        entry_point="inference.py",
        source_dir=".",
        framework_version=FRAMEWORK_VERSION,
        sagemaker_session=sm_session,
    )

    print("\nDeploying endpoint (10-15 minutes)...")
    predictor = model.deploy(
        initial_instance_count=1,
        instance_type=INSTANCE_TYPE,
        endpoint_name=ENDPOINT_NAME
    )
    
    print("Testing endpoint with sample data...")
    sample = {
        "Month": "January",
        "Age": 30,
        "Occupation": "Engineer",
        "Annual_Income": 50000,
        "Monthly_Inhand_Salary": 4000,
        "Num_Bank_Accounts": 4,
        "Num_Credit_Card": 3,
        "Interest_Rate": 10,
        "Num_of_Loan": 2,
        "Delay_from_due_date": 5,
        "Num_of_Delayed_Payment": 2,
        "Changed_Credit_Limit": 5,
        "Num_Credit_Inquiries": 3,
        "Credit_Mix": "Good",
        "Outstanding_Debt": 1000,
        "Credit_Utilization_Ratio": 30,
        "Credit_History_Age": 60,
        "Payment_of_Min_Amount": "No",
        "Total_EMI_per_month": 200,
        "Amount_invested_monthly": 300,
        "Payment_Behaviour": "High_spent_Small_value_payments",
        "Monthly_Balance": 500,
        "Auto Loan": 1,
        "Credit-Builder Loan": 0,
        "Debt Consolidation Loan": 0,
        "Home Equity Loan": 0,
        "Mortgage Loan": 0,
        "Not Specified": 0,
        "Payday Loan": 0,
        "Personal Loan": 0,
        "Student Loan": 0,
        "Unknown": 0
    }
    
    
    runtime = boto3.client("sagemaker-runtime", region_name=REGION)
    response = runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Accept="application/json",
        Body=json.dumps(sample),
    )
    print("\nSmoke test response:")
    print(response["Body"].read().decode("utf-8"))

    #print(
     #   f"\nEndpoint '{ENDPOINT_NAME}' is live in {REGION}.\n"
      #  f"Delete it before lab teardown: predictor.delete_endpoint()"
    #)


if __name__ == "__main__":
    main()


Role:      arn:aws:iam::472608500930:role/LabRole
Model URI: s3://sagemaker-us-east-1-472608500930/model/model.tar.gz
Endpoint:  credit-score-endpoint

Deploying endpoint (10-15 minutes)...
------!Testing endpoint with sample data...

Smoke test response:
{"prediction": 1, "label": "Standard", "probabilities": {"Poor": 0.07644087821245193, "Standard": 0.6479282379150391, "Good": 0.2756308317184448}}


In [ ]:
#check log here: https://us-east-1.console.aws.amazon.com/cloudwatch/home?utm_source=chatgpt.com&region=us-east-1#logsV2:log-groups

In [27]:
import boto3

sm_client = boto3.client("sagemaker", region_name="us-east-1")
ENDPOINT_NAME = "credit-score-endpoint"

# 1. Delete the stuck endpoint
print(f"Deleting failed endpoint: {ENDPOINT_NAME}...")
try:
    sm_client.delete_endpoint(EndpointName=ENDPOINT_NAME)
    print("Endpoint deletion triggered.")
except Exception as e:
    print(f"No endpoint found to delete: {e}")

# 2. Delete the conflicting endpoint configuration
print(f"Deleting endpoint configuration: {ENDPOINT_NAME}...")
try:
    sm_client.delete_endpoint_config(EndpointConfigName=ENDPOINT_NAME)
    print("Endpoint configuration deletion triggered.")
except Exception as e:
    print(f"No config found to delete: {e}")

print("\nCleanup complete! You can now safely run your main deploy script.")

Deleting failed endpoint: credit-score-endpoint...
No endpoint found to delete: An error occurred (ValidationException) when calling the DeleteEndpoint operation: Could not find endpoint "credit-score-endpoint".
Deleting endpoint configuration: credit-score-endpoint...
No config found to delete: An error occurred (ValidationException) when calling the DeleteEndpointConfig operation: Could not find endpoint configuration "credit-score-endpoint".

Cleanup complete! You can now safely run your main deploy script.
